In [ ]:
from pathlib import Path

from matplotlib import pyplot as plt

# The SBI Simulator

This (mostly text based) walk through will teach how to convert physics engines you've played with in part one into a simulator compatible with MaCh3SBITools

## Working with MaCh3SBITools
Now we've had a play with a "Traditional" Bayesian workflow, let's see if we can do better!
The first thing we need to do is write our own simulator class. Full details are in the [documentation](https://henry-wallace-phys.github.io/MaCh3SbiTools/modules/getting_started/building_simulator.html) but this guide will take you through step by step

[Note: This code will not be written in the notebook]

----

The first step is to open [my_simulator/my_simulator.py](my_simulator/my_simulator.py). This will be the skeleton from which we convert our parameter handler etc. into a proper simulator.

Currently, there is an empty class `MySimulator`, `MaCh3PythonTools` expected a every simulator to contain the same methods:

* `__init__` : This should take a single config file as input
*  `simulate`: This should take a set of parameter values and output the MC bins for those values
* `get_parameter_names`: This should output the names of your parameters
* `get_parameter_bounds`: This should output two lists [lower_bounds, upper_bounds]
* `get_is_flat`: This should tell you if a given parameter index is flat
* `get_data_bins`: Should give you your data bins
* `get_parameter_nominals`: Give you the nominal values of your parameters
* `get_parameter_errors`: Gives you the errors on your parameters
* `get_covariance_matrix` Gets you parameter covariance matrix

----

Now we've had a summary, let's put it into practice!

The first step is the initialiser, here we can use the tool we developed earlier to open our YAMLs with a single YAML!
Let's make it a helper function + copy/paste the Cell where we add our YAMLs earlier. MySimulator then becomes something
like

```py
class MySimulator:
    def __init__(self, config_path: Path):
        self.parameter_handler = ParameterHandler(config_path)
        sample_config = self._get_sample_yaml(config_path)
        self.sample_handler = SampleHandler(sample_config, self.parameter_handler)


    @classmethod
    def _get_sample_yamls(cls, config_path: Path):
        # First we get the config and check it exists
        physics_config = Path("physics_configs/PhysicsConfig.yaml")

        if not physics_config.is_file() and not physics_config.exists():
            raise FileNotFoundError("Config file not found.")

        # We extract the sample config from the physics config. This is a tad hacky but
        # is what's currently done in MaCh3
        with open(physics_config, "r") as f:
            physics_open = yaml.safe_load(f)
            sample_config = Path(physics_open['Sample']['sample_config'])

        # We also need to get the SAMPLE config from the physics config!
        if not sample_config.is_file():
            raise ValueError("Config file not found.")

        return sample_config
```

This will nicely initialise everything!


Most methods can then be plumbed in pretty easily

```py
class MySimulator:
    ...
    def get_parameter_names(self):
        return np.array(self.parameter_handler.get_parameter_names())

    def get_parameter_bounds(self):
        lower = []
        upper = []

        for p in self.parameter_handler.n_params:
            l, u = self.parameter_handler.get_bounds(p)
            lower.append(l)
            upper.append(u)
        return np.array(lower), np.array(upper)

    def get_parameter_nominals(self):
        return np.array(self.parameter_handler.nominal_values)

    def get_parameter_errors(self):
        return np.array([self.parameter_handler.get_error(i) for i in range(len(self.parameter_handler.n_params))])

    def get_covariance_matrix(self):
        return self.parameter_handler.covariance_matrix

    def get_is_flat(self, i: int):
        return self.parameter_handler.get_is_flat(i)

    def get_data_bins(self):
        return self.sample_handler.get_data()
```

The final, most important, step is adding in the `simulate` method.

### The simulate method
Simulate here is a 3 stage process
1. Set my parameters to some value
2. Reweight my sample handler
3. Get my MC prediction

But we've already done this! This means that the simulate method is simply
```py
    def simulate(self, theta):
        self.parameter_handler.set_parameter_values(theta)
        self.sample_handler.reweight()
        return self.sample_handler.get_mc_vals()
```

Now let's check your implementation!

### MaCh3 SBI Tools Simulator

We can now use the MaCh3 SBI tools features with your simulator wrapper! The basic object we care about here is the Simulator. Let's load your config into the simulator

In [ ]:
# First things first, let's use the MaCh3SBITools Simulator instance
from mach3sbitools.simulator import Simulator

# Our config file
physics_config = Path("physics_configs/PhysicsConfig.yaml")

# Here the arguments are
# 1. The python import path to your simulator module
# 2. The simulator class name in that modules
# 3. The config

# We also let it know that our cyclical parameter is cyclical!
mach3sbi_simulator = Simulator(
    "my_simulator", "MySimulator", physics_config, cyclical_pars=["cyclical_param"]
)

In [ ]:
# Now we have a simulator we can draw samples from so let's do that! Here we take 10000 samples
theta, x = mach3sbi_simulator.simulate(10000)

In [ ]:
# We can use SBI's plotting utils to plot them!
# We can see our priors!
from sbi.analysis import pairplot as sbi_pairplot

sbi_pairplot(theta)
plt.show()

Now we've had a play with the simulator and built our own wrapper, let's start using the "proper" workflow!